In [1]:
import xarray as xr
import numpy as np
from pathlib import Path

BASE_DIR = Path.cwd()
RAW_DIR = BASE_DIR / "Crops" / "a_raw" / "CROPGRIDSv1.08_NC_maps"
OUT_DIR = BASE_DIR / "Crops" / "b_processed" / "bronze" / "top10_maps"
OUT_DIR.mkdir(parents=True, exist_ok=True)

EGYPT_CODE = 57

# Load country mask once
countries = xr.open_dataset(RAW_DIR / "Countries_2018.nc")
mask = countries["country"] == EGYPT_CODE

results = []

# Loop through all crop files
for file in RAW_DIR.glob("CROPGRIDSv1.08_*.nc"):
    if file.name == "Countries_2018.nc":
        continue

    crop_name = file.stem.replace("CROPGRIDSv1.08_", "")
    print("Processing:", crop_name)

    ds = xr.open_dataset(file)

    if "harvarea" not in ds:
        ds.close()
        continue

    harv = ds["harvarea"].where(mask)
    vals = harv.values

    total_area = float(np.nansum(vals[vals > 0]))
    cells = int(np.sum(vals > 0))
    max_val = float(np.nanmax(vals)) if np.any(vals > 0) else 0.0

    if total_area > 0:
        results.append({
            "crop": crop_name,
            "total_harvarea_ha": total_area,
            "cells_with_crop": cells,
            "max_ha_per_cell": max_val
        })

    ds.close()

countries.close()

# Sort by cultivated area
results = sorted(results, key=lambda x: x["total_harvarea_ha"], reverse=True)

Processing: abaca
Processing: agave
Processing: alfalfa
Processing: almond
Processing: aniseetc
Processing: apple
Processing: apricot
Processing: areca
Processing: artichoke
Processing: asparagus
Processing: avocado
Processing: bambara
Processing: banana
Processing: barley
Processing: bean
Processing: beetfor
Processing: berrynes
Processing: blueberry
Processing: brazil
Processing: broadbean
Processing: buckwheat
Processing: cabbage
Processing: cabbagefor
Processing: canaryseed
Processing: carob
Processing: carrot
Processing: carrotfor
Processing: cashew
Processing: cashewapple
Processing: cassava
Processing: castor
Processing: cauliflower
Processing: cerealnes
Processing: cherry
Processing: chestnut
Processing: chickpea
Processing: chicory
Processing: chilleetc
Processing: cinnamon
Processing: citrusnes
Processing: clove
Processing: clover
Processing: cocoa
Processing: coconut
Processing: coffee
Processing: cotton
Processing: cowpea
Processing: cranberry
Processing: cucumberetc
Proces

In [2]:
# Print ranking
print("\n" + "=" * 70)
print("CROPS IN EGYPT — RANKED BY HARVESTED AREA")
print("=" * 70)
print(f"{'Rank':<6}{'Crop':<30}{'Total area (ha)':>18}{'Cells':>10}{'Max/cell':>14}")
print("-" * 70)

for i, r in enumerate(results, start=1):
    print(f"{i:<6}{r['crop']:<30}{r['total_harvarea_ha']:>18,.0f}{r['cells_with_crop']:>10}{r['max_ha_per_cell']:>14.1f}")


CROPS IN EGYPT — RANKED BY HARVESTED AREA
Rank  Crop                             Total area (ha)     Cells      Max/cell
----------------------------------------------------------------------
1     maize                                  1,049,519      2252        1809.9
2     rice                                     510,439      1679        1466.8
3     wheat                                    487,052      2200        1580.1
4     sorghum                                  126,828      1071         824.6
5     potato                                   119,339      1462         794.7
6     tomato                                   105,344      2245         186.7
7     sugarbeet                                 97,427      1295         402.4
8     sugarcane                                 83,429      2074         109.8
9     vegfor                                    83,260      2286         102.3
10    mango                                     82,381      2178         233.0
11    orange     

In [3]:
# Export maps for top 10 crops
print("\n" + "=" * 70)
print("EXPORTING TOP 10 MAPS")
print("=" * 70)

# Re-open country mask because it was closed above
countries = xr.open_dataset(RAW_DIR / "Countries_2018.nc")
mask = countries["country"] == EGYPT_CODE

top10 = results[:10]

for r in top10:
    crop = r["crop"]
    print("Exporting:", crop)

    ds = xr.open_dataset(RAW_DIR / f"CROPGRIDSv1.08_{crop}.nc")

    harv = ds["harvarea"].where(mask, drop=True)
    harv = harv.where(harv > 0, other=-9999).astype("float32")

    harv = harv.rio.set_spatial_dims(x_dim="lon", y_dim="lat")
    harv = harv.rio.write_crs("EPSG:4326")
    harv = harv.rio.write_nodata(-9999)

    out_path = OUT_DIR / f"{crop}_harvarea_egypt.tif"
    harv.rio.to_raster(out_path)

    print("  Saved:", out_path)
    ds.close()

countries.close()
print("\nDone.")


EXPORTING TOP 10 MAPS
Exporting: maize


AttributeError: 'DataArray' object has no attribute 'rio'